In [ ]:
# ================================================
# SILVER LAYER
# ================================================
import os
from datetime import datetime

BRONZE_PATH = "/lakehouse/default/Files/bronze"
SILVER_PATH = "/lakehouse/default/Files/silver"

# Create silver folder
os.makedirs(SILVER_PATH, exist_ok=True)

print("Silver paths ready ")
print(f"\nBronze tables available:")
for folder in sorted(os.listdir(BRONZE_PATH)):
    print(f"   {folder}")


In [ ]:
import os

silver_path = "/lakehouse/default/Files/silver"

print("Silver tables found:")
items = os.listdir(silver_path)

if items:
    for item in sorted(items):
        print(f"   {item}")
else:
    print("   Empty — no tables yet")


In [5]:
import pandas as pd
import pyarrow as pa
import pyarrow.parquet as pq
from datetime import datetime
import os

BRONZE_PATH = "/lakehouse/default/Files/bronze"
SILVER_PATH = "/lakehouse/default/Files/silver"

# ── 1. PATIENTS ──────────────────────────────
df = pd.read_parquet(f"{BRONZE_PATH}/patients/part-0.parquet")
df = df.drop_duplicates(subset=["patient_id"])
df["dob"] = pd.to_datetime(df["dob"], dayfirst=True, errors="coerce")
df["registration_date"] = pd.to_datetime(df["registration_date"], dayfirst=True, errors="coerce")
df["gender"] = df["gender"].str.strip().str.title()
df["ethnicity"] = df["ethnicity"].str.strip().str.title()
df["insurance_type"] = df["insurance_type"].str.strip().str.upper()
df["state"] = df["state"].str.strip().str.upper()
df["city"] = df["city"].str.strip().str.title()
df["phone"] = df["phone"].fillna("Unknown")
df["email"] = df["email"].fillna("Unknown")
df["_silver_load_timestamp"] = datetime.now().isoformat()
df = df.drop(columns=["_bronze_load_timestamp", "_source_file"])
os.makedirs(f"{SILVER_PATH}/patients", exist_ok=True)
pq.write_table(pa.Table.from_pandas(df), f"{SILVER_PATH}/patients/part-0.parquet")
print(f" silver/patients — {len(df):,} rows")

# ── 2. ENCOUNTERS ─────────────────────────────
df = pd.read_parquet(f"{BRONZE_PATH}/encounters/part-0.parquet")
df = df.drop_duplicates(subset=["encounter_id"])
df["visit_date"] = pd.to_datetime(df["visit_date"], dayfirst=True, errors="coerce")
df["discharge_date"] = pd.to_datetime(df["discharge_date"], dayfirst=True, errors="coerce")
df["visit_type"] = df["visit_type"].str.strip().str.title()
df["department"] = df["department"].str.strip().str.title()
df["status"] = df["status"].str.strip().str.title()
df["admission_type"] = df["admission_type"].fillna("Not Applicable")
df["length_of_stay"] = df["length_of_stay"].fillna(0)
df["readmitted_flag"] = df["readmitted_flag"].fillna("No")
df["_silver_load_timestamp"] = datetime.now().isoformat()
df = df.drop(columns=["_bronze_load_timestamp", "_source_file"])
os.makedirs(f"{SILVER_PATH}/encounters", exist_ok=True)
pq.write_table(pa.Table.from_pandas(df), f"{SILVER_PATH}/encounters/part-0.parquet")
print(f" silver/encounters — {len(df):,} rows")

# ── 3. PROVIDERS ──────────────────────────────
df = pd.read_parquet(f"{BRONZE_PATH}/providers/part-0.parquet")
df = df.drop_duplicates(subset=["provider_id"])
df["specialty"] = df["specialty"].str.strip().str.title()
df["department"] = df["department"].str.strip().str.title()
df["inhouse"] = df["inhouse"].str.strip().str.title()
df["email"] = df["email"].fillna("Unknown")
df["_silver_load_timestamp"] = datetime.now().isoformat()
df = df.drop(columns=["_bronze_load_timestamp", "_source_file"])
os.makedirs(f"{SILVER_PATH}/providers", exist_ok=True)
pq.write_table(pa.Table.from_pandas(df), f"{SILVER_PATH}/providers/part-0.parquet")
print(f" silver/providers — {len(df):,} rows")

# ── 4. CLAIMS & BILLING ───────────────────────
df = pd.read_parquet(f"{BRONZE_PATH}/claims_and_billing/part-0.parquet")
df = df.drop_duplicates(subset=["billing_id"])
df["claim_billing_date"] = pd.to_datetime(df["claim_billing_date"], errors="coerce")
df["claim_status"] = df["claim_status"].str.strip().str.title()
df["insurance_provider"] = df["insurance_provider"].str.strip().str.upper()
df["payment_method"] = df["payment_method"].str.strip().str.title()
df["denial_reason"] = df["denial_reason"].fillna("No Denial")
df["paid_amount"] = df["paid_amount"].fillna(0)
df["unpaid_amount"] = df["billed_amount"] - df["paid_amount"]
df["collection_rate"] = (df["paid_amount"] / df["billed_amount"] * 100).round(2)
df["_silver_load_timestamp"] = datetime.now().isoformat()
df = df.drop(columns=["_bronze_load_timestamp", "_source_file"])
os.makedirs(f"{SILVER_PATH}/claims_and_billing", exist_ok=True)
pq.write_table(pa.Table.from_pandas(df), f"{SILVER_PATH}/claims_and_billing/part-0.parquet")
print(f" silver/claims_and_billing — {len(df):,} rows")

# ── 5. DENIALS ────────────────────────────────
df = pd.read_parquet(f"{BRONZE_PATH}/denials/part-0.parquet")
df = df.drop_duplicates(subset=["denial_id"])
df["denial_date"] = pd.to_datetime(df["denial_date"], dayfirst=True, errors="coerce")
df["appeal_resolution_date"] = pd.to_datetime(df["appeal_resolution_date"], dayfirst=True, errors="coerce")
df["appeal_filed"] = df["appeal_filed"].str.strip().str.title()
df["appeal_status"] = df["appeal_status"].fillna("Not Filed")
df["final_outcome"] = df["final_outcome"].fillna("Pending")
df["_silver_load_timestamp"] = datetime.now().isoformat()
df = df.drop(columns=["_bronze_load_timestamp", "_source_file"])
os.makedirs(f"{SILVER_PATH}/denials", exist_ok=True)
pq.write_table(pa.Table.from_pandas(df), f"{SILVER_PATH}/denials/part-0.parquet")
print(f" silver/denials — {len(df):,} rows")

# ── 6. DIAGNOSES ──────────────────────────────
df = pd.read_parquet(f"{BRONZE_PATH}/diagnoses/part-0.parquet")
df = df.drop_duplicates(subset=["diagnosis_id", "encounter_id"])
df["diagnosis_description"] = df["diagnosis_description"].str.strip().str.title()
df["primary_flag"] = df["primary_flag"].fillna(False)
df["chronic_flag"] = df["chronic_flag"].fillna(False)
df["_silver_load_timestamp"] = datetime.now().isoformat()
df = df.drop(columns=["_bronze_load_timestamp", "_source_file"])
os.makedirs(f"{SILVER_PATH}/diagnoses", exist_ok=True)
pq.write_table(pa.Table.from_pandas(df), f"{SILVER_PATH}/diagnoses/part-0.parquet")
print(f" silver/diagnoses — {len(df):,} rows")

# ── 7. PROCEDURES ─────────────────────────────
df = pd.read_parquet(f"{BRONZE_PATH}/procedures/part-0.parquet")
df = df.drop_duplicates(subset=["procedure_id"])
df["procedure_date"] = pd.to_datetime(df["procedure_date"], dayfirst=True, errors="coerce")
df["procedure_description"] = df["procedure_description"].str.strip().str.title()
df["procedure_cost"] = df["procedure_cost"].fillna(0)
df["_silver_load_timestamp"] = datetime.now().isoformat()
df = df.drop(columns=["_bronze_load_timestamp", "_source_file"])
os.makedirs(f"{SILVER_PATH}/procedures", exist_ok=True)
pq.write_table(pa.Table.from_pandas(df), f"{SILVER_PATH}/procedures/part-0.parquet")
print(f" silver/procedures — {len(df):,} rows")

# ── 8. MEDICATIONS ────────────────────────────
df = pd.read_parquet(f"{BRONZE_PATH}/medications/part-0.parquet")
df = df.drop_duplicates(subset=["medication_id"])
df["prescribed_date"] = pd.to_datetime(df["prescribed_date"], dayfirst=True, errors="coerce")
df["drug_name"] = df["drug_name"].str.strip().str.title()
df["route"] = df["route"].str.strip().str.title()
df["cost"] = df["cost"].fillna(0)
df["_silver_load_timestamp"] = datetime.now().isoformat()
df = df.drop(columns=["_bronze_load_timestamp", "_source_file"])
os.makedirs(f"{SILVER_PATH}/medications", exist_ok=True)
pq.write_table(pa.Table.from_pandas(df), f"{SILVER_PATH}/medications/part-0.parquet")
print(f" silver/medications — {len(df):,} rows")

# ── 9. LAB TESTS ──────────────────────────────
df = pd.read_parquet(f"{BRONZE_PATH}/lab_tests/part-0.parquet")
df = df.drop_duplicates(subset=["lab_id"])
df["test_date"] = pd.to_datetime(df["test_date"], dayfirst=True, errors="coerce")
df["test_name"] = df["test_name"].str.strip().str.title()
df["status"] = df["status"].str.strip().str.title()
df["test_result"] = df["test_result"].fillna("Pending")
df["_silver_load_timestamp"] = datetime.now().isoformat()
df = df.drop(columns=["_bronze_load_timestamp", "_source_file"])
os.makedirs(f"{SILVER_PATH}/lab_tests", exist_ok=True)
pq.write_table(pa.Table.from_pandas(df), f"{SILVER_PATH}/lab_tests/part-0.parquet")
print(f" silver/lab_tests — {len(df):,} rows")

print("\n Silver layer complete!")




 silver/patients — 60,000 rows
 silver/encounters — 70,000 rows
 silver/providers — 1,491 rows
 silver/claims_and_billing — 70,000 rows
 silver/denials — 5,998 rows
 silver/diagnoses — 70,000 rows
 silver/procedures — 138 rows
 silver/medications — 52,500 rows
 silver/lab_tests — 17 rows

 Silver layer complete!


In [12]:
#review tables with wrong rows numbers
# Check what's happening with these 3 tables
for table_name, pk in [
    ("diagnoses", "diagnosis_id"),
    ("procedures", "procedure_id"),
    ("lab_tests", "lab_id")
]:
    df = pd.read_parquet(
        f"/lakehouse/default/Files/bronze/{table_name}/part-0.parquet"
    )
    print(f"\n {table_name}")
    print(f"   Total rows:        {len(df):,}")
    print(f"   Null PKs:          {df[pk].isna().sum():,}")
    print(f"   Duplicate PKs:     {df[pk].duplicated().sum():,}")
    print(f"   Unique PKs:        {df[pk].nunique():,}")
    print(f"   Sample PKs: {df[pk].head(3).tolist()}")



 diagnoses
   Total rows:        70,000
   Null PKs:          0
   Duplicate PKs:     69,937
   Unique PKs:        63
   Sample PKs: ['DIA0024', 'DIA0041', 'DIA0036']

 procedures
   Total rows:        126,021
   Null PKs:          0
   Duplicate PKs:     125,883
   Unique PKs:        138
   Sample PKs: ['PROC00009', 'PROC00012', 'PROC00024']

 lab_tests
   Total rows:        54,537
   Null PKs:          0
   Duplicate PKs:     54,520
   Unique PKs:        17
   Sample PKs: ['LAB010', 'LAB010', 'LAB001']


In [13]:
#Use Correct Primary Keys

SILVER_PATH = "/lakehouse/default/Files/silver"
BRONZE_PATH = "/lakehouse/default/Files/bronze"

# ── DIAGNOSES ──
# Real unique key = diagnosis_id + encounter_id combined
df = pd.read_parquet(f"{BRONZE_PATH}/diagnoses/part-0.parquet")
print(f"Before: {len(df):,}")
df = df.drop_duplicates(subset=["diagnosis_id", "encounter_id"])
print(f"After dedup: {len(df):,}")
df["diagnosis_description"] = df["diagnosis_description"].str.strip().str.title()
df["primary_flag"] = df["primary_flag"].fillna(False)
df["chronic_flag"] = df["chronic_flag"].fillna(False)
df["_silver_load_timestamp"] = datetime.now().isoformat()
df = df.drop(columns=["_bronze_load_timestamp", "_source_file"])
os.makedirs(f"{SILVER_PATH}/diagnoses", exist_ok=True)
pq.write_table(pa.Table.from_pandas(df), f"{SILVER_PATH}/diagnoses/part-0.parquet")
print(f" silver/diagnoses — {len(df):,} rows")

# ── PROCEDURES ──
# Real unique key = procedure_id + encounter_id combined
df = pd.read_parquet(f"{BRONZE_PATH}/procedures/part-0.parquet")
print(f"\nBefore: {len(df):,}")
df = df.drop_duplicates(subset=["procedure_id", "encounter_id"])
print(f"After dedup: {len(df):,}")
df["procedure_date"] = pd.to_datetime(df["procedure_date"], dayfirst=True, errors="coerce")
df["procedure_description"] = df["procedure_description"].str.strip().str.title()
df["procedure_cost"] = df["procedure_cost"].fillna(0)
df["_silver_load_timestamp"] = datetime.now().isoformat()
df = df.drop(columns=["_bronze_load_timestamp", "_source_file"])
os.makedirs(f"{SILVER_PATH}/procedures", exist_ok=True)
pq.write_table(pa.Table.from_pandas(df), f"{SILVER_PATH}/procedures/part-0.parquet")
print(f" silver/procedures — {len(df):,} rows")

# ── LAB TESTS ──
# Real unique key = lab_id + encounter_id combined
df = pd.read_parquet(f"{BRONZE_PATH}/lab_tests/part-0.parquet")
print(f"\nBefore: {len(df):,}")
df = df.drop_duplicates(subset=["lab_id", "encounter_id"])
print(f"After dedup: {len(df):,}")
df["test_date"] = pd.to_datetime(df["test_date"], dayfirst=True, errors="coerce")
df["test_name"] = df["test_name"].str.strip().str.title()
df["status"] = df["status"].str.strip().str.title()
df["test_result"] = df["test_result"].fillna("Pending")
df["_silver_load_timestamp"] = datetime.now().isoformat()
df = df.drop(columns=["_bronze_load_timestamp", "_source_file"])
os.makedirs(f"{SILVER_PATH}/lab_tests", exist_ok=True)
pq.write_table(pa.Table.from_pandas(df), f"{SILVER_PATH}/lab_tests/part-0.parquet")
print(f" silver/lab_tests — {len(df):,} rows")

print("\n 3 tables fixed!")


Before: 70,000
After dedup: 70,000
 silver/diagnoses — 70,000 rows

Before: 126,021
After dedup: 126,021
 silver/procedures — 126,021 rows

Before: 54,537
After dedup: 51,565
 silver/lab_tests — 51,565 rows

 3 tables fixed!


In [15]:
# verification
import os

SILVER_PATH = "/lakehouse/default/Files/silver"

print("=" * 50)
print(f"{'Table':<35} {'Rows':>12}")
print("=" * 50)

for folder in sorted(os.listdir(SILVER_PATH)):
    df_check = pd.read_parquet(
        f"{SILVER_PATH}/{folder}/part-0.parquet"
    )
    print(f"{('silver/'+folder):<35} {len(df_check):>12,}")

print("=" * 50)
print(" Silver verification complete!")


Table                                       Rows
silver/claims_and_billing                 70,000
silver/denials                             5,998
silver/diagnoses                          70,000
silver/encounters                         70,000
silver/lab_tests                          51,565
silver/medications                        52,500
silver/patients                           60,000
silver/procedures                        126,021
silver/providers                           1,491
 Silver verification complete!


In [23]:
# CHECK
#Options:
# patients, encounters, providers
# claims_and_billing, denials, diagnoses
# procedures, medications, lab_tests

df_check = pd.read_parquet(
    "/lakehouse/default/Files/silver/patients/part-0.parquet"
)
df_check.head(10)


,patient_id,first_name,last_name,dob,age,gender,ethnicity,insurance_type,marital_status,address,city,state,zip,phone,email,registration_date,_silver_load_timestamp
0,PAT000001,Danielle,Johnson,1940-05-14,85,Female,Asian,UHC,Married,32181 Johnson Course Apt. 389,Bakersfield,CA,93301.0,Unknown,danielle.johnson40@example.com,2025-01-03,2026-06-04T16:38:46.451562
1,PAT000002,Anna,Baldwin,2010-03-04,15,Female,White,AETNA,Married,79402 Peterson Drives Apt. 511,Bakersfield,CA,93301.0,(615)759 407,anna.baldwin294@example.com,2025-02-26,2026-06-04T16:38:46.451562
2,PAT000003,James,Jones,2021-03-31,4,Male,White,UHC,Married,1316 Chavez Village,Portland,OR,97035.0,(925)853 419,james.jones377@example.com,2025-01-27,2026-06-04T16:38:46.451562
3,PAT000004,Veronica,Bowman,1976-08-26,48,Female,Hispanic,BCBS,Widowed/Divorced/Separated,503 Linda Locks,Seattle,WA,98101.0,Unknown,veronica.bowman653@example.com,2025-01-26,2026-06-04T16:38:46.451562
4,PAT000005,Carl,Gentry,1942-01-06,83,Male,Hispanic,BCBS,Married,None,None,None,NaN,001 553 676,carl.gentry91@example.com,2025-03-14,2026-06-04T16:38:46.451562
5,PAT000006,Brenda,Hurst,1992-05-07,33,Female,Hispanic,HUMANA,Single,3287 Katelyn Wall Apt. 226,Sacramento,CA,94203.0,+1 616 769 7,brenda.hurst969@example.com,2025-01-11,2026-06-04T16:38:46.451562
6,PAT000007,Kelly,Moore,1939-06-05,86,Male,Asian,MEDICAID,Widowed/Divorced/Separated,482 Monica Hills,Long Beach,CA,90802.0,+1 289 332 5,kelly.moore714@example.com,2025-02-05,2026-06-04T16:38:46.451562
7,PAT000008,Natasha,Decker,1950-03-18,75,Male,White,CIGNA,Unknown,03911 Cabrera Trace Apt. 278,Los Angeles,CA,90001.0,(848)796 383,natasha.decker77@example.com,2025-02-12,2026-06-04T16:38:46.451562
8,PAT000009,Rachel,Hayes,1993-09-10,31,Male,White,MEDICAID,Single,15098 Brianna Avenue,Seattle,WA,98101.0,603 910 5183,Unknown,2025-01-21,2026-06-04T16:38:46.451562
9,PAT000010,Amber,Cummings,1949-10-06,75,Female,Asian,AETNA,Widowed/Divorced/Separated,76311 Gomez Loop Suite 010,San Francisco,CA,94101.0,(551)333 387,Unknown,2025-02-13,2026-06-04T16:38:46.451562
